<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/2.3_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

## Creating subagents

In [2]:
import os
from langchain import chat_models

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [3]:
from langchain import tools, agents

@tools.tool
def square_root(x: float) -> float:
    """
    Calculate the square root of a number
    """
    return x**0.5

# create subagents
subagent_1 = agents.create_agent(
    model=llm,
    tools=[square_root]
)

@tools.tool
def square(x: float) -> float:
    """
    Calculate the square of a number
    """
    return x**2

subagent_2 = agents.create_agent(
    model=llm,
    tools=[square]
)

## Calling subagents

In [4]:
from langchain import tools, messages, agents

@tools.tool
def call_subagent_1(x: float) -> float:
    """
    Call subagent 1 in order to calculate the square root of a number
    """
    response = subagent_1.invoke(
        input={"messages": [messages.HumanMessage(content=f"""
            Calculate the square root of {x}
        """)]}
    )
    return response["messages"][-1].content

@tools.tool
def call_subagent_2(x: float) -> float:
    """
    Call subagent 2 in order to calculate the square of a number
    """
    response = subagent_2.invoke(
        input={"messages": [messages.HumanMessage(content=f"""
            Calculate the square of {x}
        """)]}
    )
    return response["messages"][-1].content

## Creating the main agent
main_agent = agents.create_agent(
    model=llm,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="""
        You are a helpful assistant who can call subagents
        to calculate the square root or square of a number.
    """
)

## Test

In [5]:
import time
from langchain import messages

question = "What is the square root of 456?"
start_time = time.time()
response = main_agent.invoke(
    input={"messages": [messages.HumanMessage(content=question)]}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

Time taken: 3.59s


In [6]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is the square root of 456?
================================== Ai Message ==================================
Tool Calls:
  call_subagent_1 (call_q3hsof58)
 Call ID: call_q3hsof58
  Args:
    x: 456
================================= Tool Message =================================
Name: call_subagent_1

The square root of 456.0 is approximately **21.3542**.
================================== Ai Message ==================================

The square root of 456 is approximately **21.3542**.
